<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Building the Dataset

9,523 cancer pathology reports live in one file. The cancer type each one belongs to
lives in a different file. Neither is any use alone: to teach a computer to read a
report we need the report and its answer sitting side by side, on the same row.

That is the whole lab. Download both files, glue them together on the patient barcode,
and look at what we ended up with.

**Puzzles.** Cells marked **PUZZLE** have blanks written as `___`. Fill them in and run
the cell. Each puzzle ends with an `assert`, which stays silent when you are right and
stops the cell when you are wrong. Every puzzle points at the appendix section that
covers it.

## Setup

In [ ]:
# pandas reads a CSV straight from a web address, so there is nothing to install
# and nothing to download by hand. This works the same in Colab and on your laptop.
import pandas as pd

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

## The reports

One row per patient. The file is about 35 MB, so the next cell takes a few seconds.

In [ ]:
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")

# .shape is (rows, columns).
reports.shape

In [ ]:
# Always look at data before doing anything to it.
reports.head()

In [ ]:
# One whole report, so you know what we are really working with.
print(reports.loc[0, "text"][:400])

## The barcode

`patient_filename` is two things stuck together with a dot: the patient's barcode, then
a long random report id.

    TCGA-BP-5195.25c0b433-5557-4165-922e-2c1eac9c26f0
    ^^^^^^^^^^^^
    this part identifies the patient

The barcode is the only thing the two files have in common, so everything below depends
on getting it out.

In [ ]:
# PUZZLE 1 — cut the filename at the dot and keep the piece in front.
# Appendix A, "Splitting and joining".

reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(___)[___])

assert reports.loc[0, "patient_id"] == "TCGA-BP-5195"
reports["patient_id"].head()

In [ ]:
# PUZZLE 2 — a barcode is only safe to match on if no patient appears twice.
# Appendix B, "Checking for duplicates". Two method names, in order.

assert not reports["patient_id"].___().___()
print("One row per patient — safe to match on.")

## The labels

The cancer type is not written in the report file. It comes from TCGA's clinical
records — a separate file, from a separate source, covering a different set of patients.

In [ ]:
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")

labels.head()

In [ ]:
# 11,160 patients here against 9,523 reports. The extras are simply never matched.
labels.shape

## The join

A DataFrame's **index** is its row labels. Label both tables by barcode and `.loc` can
pull the right label for each report, however differently the two files happen to be
ordered.

In [ ]:
# PUZZLE 3 — label both tables by barcode, then look the labels up.
# Appendix B, "The index, and joining without merge".

reports.index = reports["patient_id"].values
labels.index = labels[___].values

reports["cancer_type"] = labels.loc[___, "cancer_type"]

assert reports["cancer_type"].notna().all()
reports[["patient_id", "cancer_type"]].head()

`BRCA` and `KIRC` are TCGA's shorthand. A third file spells them out.

In [ ]:
# Note sep=";" — this file is semicolon-separated. Always check the separator.
names = pd.read_csv(f"{BASE}/cancer_type/tcga-tumor-types.csv", sep=";")

names.head()

In [ ]:
# PUZZLE 4 — pair the two columns into a lookup: code -> full name.
# Appendix A, "Building and inverting dictionaries". Use the column names above.

code_to_name = dict(zip(names[___], names[___]))

assert code_to_name["KIRC"] == "Kidney renal clear cell carcinoma"
print(code_to_name["BRCA"])

In [ ]:
# Every code in our data must exist in the lookup, or the next line would fail
# partway through.
assert reports["cancer_type"].isin(code_to_name).all()

reports["cancer_type_name"] = reports["cancer_type"].apply(lambda c: code_to_name[c])
reports[["patient_id", "cancer_type", "cancer_type_name"]].head()

## How long is a report?

Worth knowing before choosing a model: some methods have a hard limit on how much text
they can read at once.

In [ ]:
# PUZZLE 5 — count the words in each report. Two blanks: a built-in function,
# and a string method. Appendix B, "Assignment creates a column"; Appendix A, .split().

reports["n_words"] = reports["text"].apply(lambda t: ___(t.___()))

assert reports["n_words"].sum() > 5_000_000
reports["n_words"].describe().round(1)

Half the reports are under ~430 words, but the longest runs to a few thousand. Keep that
spread in mind later.

## What are we up against?

Before building anything, find the score you have to beat: how often you would be right
by ignoring the text entirely and always guessing the most common cancer type. A model
that cannot beat this has learned nothing. Make this the first thing you compute, every
time.

In [ ]:
# PUZZLE 6 — count the classes, then get the largest share.
# Appendix B, "value_counts".

counts = reports["cancer_type"].___()
print(counts.head())
print("Number of classes:", len(counts))

baseline = reports["cancer_type"].value_counts(___=True).max()
print("Always-guess-BRCA baseline:", round(baseline, 3))

In [ ]:
counts.plot(kind="bar", figsize=(10, 3), title="Reports per cancer type")

32 classes, and the biggest one covers about 11% of the reports. So anything above 11%
is progress, and a lot of these classes are rare enough that a few mistakes look much
worse in the numbers than they feel.

## Save it

Text and answer on the same row, plus the two extra columns we built. This file is where
Lab 2 begins.

In [ ]:
reports[["patient_id", "text", "cancer_type", "cancer_type_name", "n_words"]].to_csv(
    "lab1_dataset.csv", index=False
)

# On Colab this file lives in the session only — download it from the file browser
# on the left if you want to keep it.
pd.read_csv("lab1_dataset.csv").head()

## What you did

- Read three files that were never designed to be used together.
- Found the one column they share, and checked it was safe to trust before trusting it.
- Joined them on it, so every report now carries its own answer.
- Measured the thing you have to beat.

Not one line of that was machine learning, and none of it was optional. Next lab: turn
the text into numbers.